# W7｜監督式學習 – Scikit-learn
**智慧醫療與機器學習基礎**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源　課程助教：古珉瑄（醫資所，m610115009@tmu.edu.tw）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/FIHML/blob/main/notebooks/W07_sklearn.ipynb)

### 今天的情境
機構每個月都有新住民入住。失智症的確診要轉介神經科，往往要等上好幾週。主任問：
> 「能不能用入住時就有的評估資料，先找出可能失智的住民，優先安排轉介？」

今天用過去 900 位住民的資料，訓練三種模型回答這個問題：**決策樹、隨機森林、羅吉斯回歸**。

> ⚠️ 本資料為**教學用模擬資料**，不對應任何真實個人。資料欄位說明見 [長照常用評估量表說明（PDF）](https://github.com/wang3yuan/FIHML/blob/main/docs/W05_長照評估量表說明.pdf)。
> ⚠️ 這裡的模型是**篩檢輔助**，不是診斷。確診仍需要醫師評估。

### 執行環境
本教材已在 **Python 3.12＋scikit-learn 1.6.1／pandas 2.2.3** 與 **scikit-learn 1.9.1／pandas 3.0.6** 兩組版本測試通過。執行下面第一格會印出你目前的版本。
不同版本的 scikit-learn 手冊內容會有些不同，**查手冊時請選擇和你相同的版本**。

### 使用方式
1. 「檔案 → 在雲端硬碟中儲存複本」，之後都在自己的複本上作業。
2. **由上往下依序執行**；看到 **▶ 實作** 的地方，請在 `# TODO` 處寫程式，或在「✏️ 你的答案」後面作答。

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Python 版本：", sys.version.split()[0])
print("scikit-learn 版本：", sklearn.__version__)
print("pandas 版本：", pd.__version__)

## 0. 設定中文字型（每次開啟 notebook 都要執行一次）
和 W6 一樣：下載思源黑體（Noto Sans TC，開源授權 OFL），讓圖上的中文正常顯示。

In [ ]:
import urllib.request
import matplotlib.font_manager as fm

FONT_URL = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/fonts/"
for name in ["NotoSansTC-Regular.ttf", "NotoSansTC-Bold.ttf"]:
    urllib.request.urlretrieve(FONT_URL + name, name)
    fm.fontManager.addfont(name)

plt.rcParams["font.family"] = "Noto Sans TC"
plt.rcParams["axes.unicode_minus"] = False   # 讓負號正常顯示
plt.rcParams["figure.dpi"] = 110

C_NO, C_YES = "#00897B", "#D06F1A"          # 非失智、失智（和 W6 相同）
print("設定完成")

## 讀取資料
W5 清理好的住民資料：每位住民一列，`dementia` 是醫師的診斷（1 = 失智、0 = 非失智）。

In [ ]:
BASE = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/data/ltc/"
df = pd.read_csv(BASE + "ltc_dementia_clean.csv")
print(df.shape)
df.head()

---
# 第 1 堂｜從資料學會預測
## 1.1 監督式學習：特徵與標籤
- **特徵（X）**：預測時拿得到的資訊，例如年齡、SPMSQ、巴氏量表……
- **標籤（y）**：想預測的答案，也就是 `dementia`

模型從「特徵＋標準答案」裡學會規則，再用在還不知道答案的新住民身上。

> ⚠️ **特徵只能用「預測當下」拿得到的資訊。** 確診之後才有的資料（CDR 分期、失智症用藥）等於偷看答案，不能放進模型。這份資料已經事先拿掉了。

💡 **認識 scikit-learn**：Python 最常用的機器學習套件之一（免費、開源，Colab 已內建）。工具依用途放在不同模組，匯入的寫法都是 `from sklearn.模組 import 名稱`：

| 模組 | 今天會用到 |
|---|---|
| `model_selection` | `train_test_split`（切分資料） |
| `tree`、`ensemble`、`linear_model` | 決策樹、隨機森林、羅吉斯回歸 |
| `neighbors`、`svm` | KNN、SVM |
| `preprocessing`、`pipeline` | `StandardScaler`（標準化）、`make_pipeline`（串接步驟） |
| `metrics` | `recall_score`、`f1_score`、`roc_auc_score`（評估指標） |

名字大寫開頭的是**模型**（先建立再使用），小寫開頭的是**函數**（直接呼叫）。

In [ ]:
data = df.dropna()                                    # 今天先把有缺值的住民拿掉（缺值的處理方法見 W5）
X = data.drop(columns=["resident_id", "dementia"])    # 特徵：住民編號不是特徵，答案也不能放進來
X = pd.get_dummies(X, drop_first=True, dtype=int)     # 文字欄位（機構、性別、血型）轉成 0／1
y = data["dementia"]                                  # 標籤：1 = 失智、0 = 非失智

print("住民數：", len(data), "　特徵數：", X.shape[1])
X.head()

## 1.2 訓練集與測試集
把住民分成兩份：
- **訓練集（75%）**：模型看得到答案，用來學規則
- **測試集（25%）**：先藏起來，最後才拿來打分數

就像考試不能直接考考古題：用學過的題目打分數，背答案也能拿 100 分。

- `stratify=y`：讓兩份的失智比例一樣
- `random_state=1151`：固定亂數，每次切出來的結果都一樣（1151 是本學期的學期代碼）

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=1151)

print("訓練集：", X_train.shape, "失智比例", round(y_train.mean(), 3))
print("測試集：", X_test.shape, "失智比例", round(y_test.mean(), 3))

## 1.3 第一個模型：決策樹
決策樹是一連串的是非題。每一步都選「讓切出來的兩群最純」的問題（**吉尼不純度**下降最多的那一刀）。

scikit-learn 的每個模型都是同樣三個步驟：**建立模型 → `fit`（學習）→ `predict`／`score`（預測、打分數）**。

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=1151)   # 1. 建立模型
tree.fit(X_train, y_train)                                       # 2. 從訓練集學規則

print("訓練集正確率：", round(tree.score(X_train, y_train), 3))  # 3. 打分數
print("測試集正確率：", round(tree.score(X_test, y_test), 3))

In [ ]:
print("模型預測：", tree.predict(X_test.head()))
print("真正答案：", y_test.head().values)

### 把樹畫出來
每個方框由上到下是：
1. **問題**（例如 `spmsq_errors <= 3.5`）：成立往左、不成立往右
2. **gini**：吉尼不純度，0 代表這群人全部同一類
3. **samples**：這個節點有幾位住民
4. **value**：[非失智人數, 失智人數]
5. **class**：多數是哪一類，走到這裡就預測成這一類

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(tree, feature_names=list(X.columns), class_names=["非失智", "失智"],
          rounded=True, fontsize=10, ax=ax)
plt.show()

第一刀切在 **SPMSQ 錯誤題數 ≤ 3.5**，也就是「錯 4 題以上」比較可能失智。這和 SPMSQ 量表本身的判讀（錯 3 題以上為認知功能障礙）很接近。
（模擬資料，只能當作示範，不能當作真實的研究發現。）

💡 `plot_tree` 也可以加上 `filled=True` 依類別上色，但預設是橘色＝第一類（非失智）、藍色＝第二類（失智），和我們「橘色＝失智」的配色規則相反，所以今天不使用。

## 1.4 樹越深越好嗎？過度擬合
樹可以一直切下去，直到每片葉子都只剩同一類的住民。這時訓練集的正確率會是 100%，但那只是**把訓練集的答案背起來**：

In [ ]:
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=1151).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, train_acc, marker="o", color="#9AA5A6", label="訓練集")
ax.plot(depths, test_acc, marker="o", color="#0B2E33", label="測試集")
ax.set_title("樹越深：訓練集越準，測試集反而變差")
ax.set_xlabel("max_depth（樹的深度）")
ax.set_ylabel("正確率")
ax.set_ylim(0.6, 1.02)
ax.legend()
plt.show()

訓練集越來越準，測試集卻在 3 層左右之後開始變差：**過度擬合（overfitting）**。
防止的方法是限制樹的大小，例如 `max_depth`（最多問幾層）、`min_samples_leaf`（每片葉子至少幾人）。這些參數的意思，第 2 堂教大家自己查手冊。

---
# 第 2 堂｜查手冊，種一片森林
## 2.1 怎麼查 scikit-learn 手冊
- 網址：<https://scikit-learn.org/stable/>，**右上角可以切換版本**（對照第一格印出的版本）
- **User Guide（使用者指南）**：觀念、數學、優缺點，例如 [1.10 Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- **API Reference（函數說明）**：每個函數的參數、屬性、方法，例如 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

API 頁面怎麼讀：
| 區塊 | 內容 | 例子 |
|---|---|---|
| 第一行 | 完整寫法：括號裡是所有參數和預設值 | `DecisionTreeClassifier(*, criterion='gini', ..., max_depth=None, ...)` |
| **Parameters** | 每個參數的型別、預設值、意思 | `max_depth : int, default=None` |
| **Attributes** | `fit` 之後才有的結果，名稱結尾有底線 `_` | `feature_importances_`、`classes_` |
| **Methods** | 可以對模型做的事 | `fit`、`predict`、`predict_proba`、`score`、`get_depth` |

在 Colab 裡也能查：在函數名稱後面加 `?` 會跳出說明；`get_params()` 會列出這個模型目前所有參數的值。

**查參數的四個步驟**：① 想清楚要調什麼 → ② 在手冊的 Parameters 找到參數 → ③ 看型別、預設值、意思 → ④ **一次只改一個**，看結果。

**常見的錯誤訊息**（先看最後一行）：
| 錯誤訊息 | 原因 | 怎麼解決 |
|---|---|---|
| `could not convert string to float` | X 裡還有文字欄位 | `pd.get_dummies()` |
| `Input X contains NaN` | 有缺值（羅吉斯回歸、KNN、SVM 不接受） | `dropna()` 或補值 |
| `Expected a 2-dimensional container` | X 只放了一欄 | `df[["age"]]`（兩層中括號） |
| `The feature names should match those that were passed during fit` | 預測時的欄位和訓練時不同 | 訓練、測試用同一套前處理 |
| `NotFittedError: ... is not fitted yet` | 還沒 `fit` 就 `predict` | 先 `fit` |
| `ConvergenceWarning: lbfgs failed to converge` | 羅吉斯回歸沒有收斂（警告） | 先標準化，或調大 `max_iter` |

In [ ]:
DecisionTreeClassifier?

In [ ]:
tree.get_params()

---
## ▶ 實作 7-1（約 10 分鐘）查手冊，調整決策樹
**(a)** 打開 [DecisionTreeClassifier 的手冊](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)，找到 `max_depth`：預設值是什麼？用預設值的話，樹會長到多深？

> ✏️ 你的答案：


**(b)** 建一棵**不設定 `max_depth`**（用預設值）的決策樹，印出訓練集與測試集的正確率。
再從手冊的 **Methods** 找出「樹的深度」和「葉子數」要用哪兩個方法，印出來。

In [ ]:
deep = DecisionTreeClassifier(random_state=1151)
# TODO：請在這裡寫程式


**(c)** 在手冊找到 `min_samples_leaf`，它是什麼意思？把它設成 `20`（其他用預設值），印出正確率、深度和葉子數，和 (b) 比較。

In [ ]:
# TODO：請在這裡寫程式


**(d)** 自己挑一組 `max_depth` 和 `min_samples_leaf`，讓**測試集正確率 ≥ 0.80**。

In [ ]:
# TODO：請在這裡寫程式


## 2.2 隨機森林：很多棵樹一起投票
一棵樹很不穩定：資料稍微不一樣，長出來的樹就不一樣。隨機森林種很多棵「不太一樣」的樹，最後投票：
1. **自助抽樣（bootstrap）**：每棵樹從訓練集「抽後放回」，抽出一樣多的住民
2. **隨機選特徵**：每次切的時候，只從隨機挑出的幾個特徵裡選
3. **投票**：每棵樹各自預測，多數決

程式和決策樹幾乎一樣，**只換了模型的名字**：

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(random_state=1151)   # 只有這一行和決策樹不同
forest.fit(X_train, y_train)

print("訓練集正確率：", round(forest.score(X_train, y_train), 3))
print("測試集正確率：", round(forest.score(X_test, y_test), 3))

訓練集 100%，測試集卻仍有約 83%：每一棵樹都過度擬合，但每棵樹錯的地方不一樣，投票之後錯誤互相抵銷。

### 特徵重要性
`feature_importances_`（結尾有底線，`fit` 之後才有）：平均下來，每個特徵讓不純度下降了多少。

In [ ]:
imp = pd.Series(forest.feature_importances_,
                index=X.columns).sort_values(ascending=False)
imp.head(10).round(3)

---
## ▶ 實作 7-2（約 12 分鐘）查手冊，調整森林
**(a)** 打開 [RandomForestClassifier 的手冊](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)：`n_estimators` 的預設值是多少？`max_features` 的預設值是什麼意思？我們有 23 個特徵，每次切的時候會看幾個？

> ✏️ 你的答案：


**(b)** 把 `n_estimators` 改成 1、10、100、500，印出測試集正確率和花費的時間。

In [ ]:
import time

for n in [1, 10, 100, 500]:
    start = time.time()
    # TODO：建立 n 棵樹的隨機森林，訓練後印出測試集正確率與花費的時間（time.time() - start）


**(c)** 用 W6 學過的 `ax.barh`，畫出前 10 名的特徵重要性（最重要的在最上面）。哪個特徵最重要？睡眠時數排第幾？

In [ ]:
# TODO：請在這裡寫程式


**(d)** 在這份模擬資料裡，身高（`height_cm`）和血型（`blood_type_*`）跟失智**完全無關**。它們的重要性分別是多少？身高為什麼不是 0？（提示：手冊裡 `feature_importances_` 的 **Warning**）

> ✏️ 你的答案：


---
# 第 3 堂｜羅吉斯回歸與模型比較
## 3.1 羅吉斯回歸：從分數到機率
名字裡有「回歸」，做的其實是**分類**：
1. 每個特徵乘上一個係數再加起來，得到分數 $z = b_0 + b_1x_1 + b_2x_2 + \cdots$
2. 用 sigmoid 把分數壓成 0 到 1 的機率：$p = \dfrac{1}{1 + e^{-z}}$
3. 機率 ≥ 0.5 就預測為失智

各特徵的單位不同（年齡 65–100 歲、跌倒 0–5 次），所以先用 `StandardScaler` **標準化**：每個特徵都變成平均 0、標準差 1。
`make_pipeline` 把標準化和模型接在一起：`fit` 時只用訓練集算平均和標準差，測試集照著用。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

logit = make_pipeline(StandardScaler(), LogisticRegression())
logit.fit(X_train, y_train)

print("訓練集正確率：", round(logit.score(X_train, y_train), 3))
print("測試集正確率：", round(logit.score(X_test, y_test), 3))

### predict_proba：每位住民失智的機率

In [ ]:
proba = logit.predict_proba(X_test.head())
print(proba.round(2))
print("模型預測：", logit.predict(X_test.head()))

### 係數：哪些特徵增加失智風險？
標準化之後，係數可以互相比較：**正的**增加失智的機率，**負的**降低；絕對值越大，影響越大。
（`logit[-1]` 是 pipeline 的最後一步，也就是羅吉斯回歸本身。）

In [ ]:
coef = pd.Series(logit[-1].coef_[0], index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(coef.index, coef.values, color=[C_YES if v > 0 else C_NO for v in coef.values])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("羅吉斯回歸的係數（標準化後）")
ax.set_xlabel("← 降低失智機率　　係數　　增加失智機率 →")
plt.show()

💡 醫學研究常用**勝算比（odds ratio）** $e^{b}$ 解讀係數：例如 SPMSQ 的係數 1.83，$e^{1.83} \approx 6.2$，代表 SPMSQ 每增加 1 個標準差（約 2.4 題），失智的勝算變成約 6.2 倍。

## 3.2 比較三個模型
正確率不夠：如果模型永遠猜「非失智」，正確率也有 56%。篩檢更在意的是：
- **敏感度**：真的失智的人，有多少被抓到？（scikit-learn 叫 **recall**）
- **特異度**：沒有失智的人，有多少被正確排除？
- **精確度**：模型說「失智」的人，有多少真的失智？（scikit-learn 叫 **precision**，醫學上叫陽性預測值 PPV）
- **F1**：精確度和敏感度的調和平均，兩個都高，F1 才會高。失智比例很低時，比正確率更不容易被騙
- **AUC**：ROC 曲線下的面積，不用選門檻，看模型整體的「排序能力」（0.5 ＝ 亂猜，1 ＝ 完美）

In [ ]:
from sklearn.metrics import recall_score, f1_score, roc_auc_score

models = {"決策樹": tree, "隨機森林": forest, "羅吉斯回歸": logit}
rows = []
for name, m in models.items():
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    rows.append([name, (pred == y_test).mean(),
                 recall_score(y_test, pred),
                 recall_score(y_test, pred, pos_label=0),
                 f1_score(y_test, pred), roc_auc_score(y_test, proba)])
cols = ["模型", "正確率", "敏感度", "特異度", "F1", "AUC"]
pd.DataFrame(rows, columns=cols).round(3)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_estimator(logit, X_test, y_test, display_labels=["非失智", "失智"],
                                      cmap="Blues", colorbar=False, ax=ax)
ax.set_xlabel("模型預測")
ax.set_ylabel("真實情況")
ax.set_title("羅吉斯回歸的混淆矩陣")
plt.show()

### ROC 曲線與 AUC
把門檻從 1 慢慢降到 0，每個門檻算一次「1 − 特異度」（誤報的比例）和敏感度，連起來就是 ROC 曲線，越靠左上角越好。
曲線下的面積就是 **AUC**：隨機挑一位失智、一位非失智的住民，模型給失智者比較高機率的機會有多大。

In [ ]:
from sklearn.metrics import roc_curve

colors = {"決策樹": "#9AA5A6", "隨機森林": "#2E6DB4", "羅吉斯回歸": "#7B4FA8"}
fig, ax = plt.subplots(figsize=(5, 5))
for name, m in models.items():
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(X_test)[:, 1])
    ax.plot(fpr, tpr, color=colors[name], linewidth=2, label=name)
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")    # 亂猜
ax.set_title("ROC 曲線：越靠左上角越好")
ax.set_xlabel("1 − 特異度（誤報的比例）")
ax.set_ylabel("敏感度（抓到失智的比例）")
ax.legend()
plt.show()

隨機森林的 AUC 最高，但在門檻 0.5 的正確率卻不是最高：AUC 看的是所有門檻的整體表現，真正使用時還是要選一個門檻。

## 3.3 調整門檻：寧可多抓，不要漏掉
三個模型的敏感度都只有約 0.70：**每 10 位失智住民，會漏掉 3 位**。
篩檢時，漏掉一位失智住民（延誤照護）的代價，通常比多轉介一位（多做一次檢查）更大。這時可以把判斷的**門檻**從 0.5 調低：

In [ ]:
p = logit.predict_proba(X_test)[:, 1]              # 每位住民失智的機率
ths = np.arange(0.05, 0.96, 0.05)
sens = [recall_score(y_test, (p >= t).astype(int)) for t in ths]
spec = [recall_score(y_test, (p >= t).astype(int), pos_label=0) for t in ths]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ths, sens, marker="o", color=C_YES, label="敏感度（抓到失智）")
ax.plot(ths, spec, marker="o", color=C_NO, label="特異度（排除非失智）")
ax.axvline(0.5, color="gray", linestyle="--")
ax.set_title("門檻越低：抓到越多失智住民，誤判也越多")
ax.set_xlabel("判斷為失智的門檻（機率）")
ax.set_ylabel("比例")
ax.legend()
plt.show()

## 3.4 其他演算法：KNN 與 SVM（簡介）
課程介紹提過的另外兩個分類演算法，用法和今天的三個模型一樣：建立 → `fit` → `predict`。
- **KNN（K 最近鄰）**：新住民來了，找特徵最像（距離最近）的 K 位住民，看他們多數有沒有失智。K 在手冊裡叫 `n_neighbors`。因為用「距離」判斷像不像，一定要先標準化。
- **SVM（支持向量機）**：在兩群住民之間找出「最寬的馬路」當分界線，壓在路邊的住民叫**支持向量**。直線分不開的資料（例如睡眠的 U 型），SVM 用**核函數（kernel）**換個角度看：例如加上「離 7 小時多遠」，就能用一條直線分開。也要先標準化。

---
## ▶ 實作 7-3（約 12 分鐘）查手冊，調整門檻
**(a)** 打開 [LogisticRegression 的手冊](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)，找到 **Methods** 裡的 `predict_proba`：它回傳什麼？第 0 欄和第 1 欄各代表什麼？

> ✏️ 你的答案：


**(b)** 用羅吉斯回歸，把門檻改成 **0.3**（機率 ≥ 0.3 就判斷為失智），算出敏感度與特異度，和門檻 0.5 比較。

In [ ]:
p = logit.predict_proba(X_test)[:, 1]
# TODO：請在這裡寫程式


**(c)** 從 3.1 的係數圖，找出**最能增加**與**最能降低**失智機率的特徵各 2 個。合理嗎？

> ✏️ 你的答案：


**(d)** 🏆 挑戰：在手冊找到 [`KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)（KNN）或 [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)（SVM），選一個用 `make_pipeline(StandardScaler(), ...)` 換掉模型。測試集正確率是多少？KNN 的 `n_neighbors`、SVM 的 `kernel` 預設值是什麼？

In [ ]:
# TODO：請在這裡寫程式


## 3.5 睡眠之謎（W9 揭曉）
W6 看到睡眠和失智是 **U 型**關係：睡太少、睡太多的住民，失智比例都比較高。兩個模型怎麼看睡眠？

In [ ]:
print("隨機森林：睡眠時數的重要性排第", list(imp.index).index("sleep_hours") + 1, "名")
print("羅吉斯回歸：睡眠時數的係數", round(coef["sleep_hours"], 2))

- **隨機森林**可以切兩刀（睡太少一刀、睡太多一刀），所以抓得到 U 型
- **羅吉斯回歸**的分數是「一條直線」：只能表示「睡越多越危險」或「睡越少越危險」，抓不到 U 型（除非自己加上「離 7 小時多遠」這種特徵，就像 SVM 的核函數）

W9 會用 **SHAP** 打開模型，看睡多、睡少各自怎麼影響每一位住民的預測。

---
### 本週 scikit-learn 指令總整理
| 目的 | 指令 |
|---|---|
| 切分資料 | `train_test_split(X, y, test_size=0.25, stratify=y, random_state=1151)` |
| 決策樹 | `DecisionTreeClassifier(max_depth=, min_samples_leaf=)`、`plot_tree()` |
| 隨機森林 | `RandomForestClassifier(n_estimators=)`、`.feature_importances_` |
| 羅吉斯回歸 | `make_pipeline(StandardScaler(), LogisticRegression())`、`[-1].coef_` |
| KNN、SVM | `make_pipeline(StandardScaler(), KNeighborsClassifier())`、`make_pipeline(StandardScaler(), SVC())` |
| 三個步驟 | `.fit(X_train, y_train)` → `.predict(X)`／`.predict_proba(X)`／`.score(X, y)` |
| 評估 | `recall_score(y, pred)`（敏感度）、`recall_score(y, pred, pos_label=0)`（特異度）、`f1_score(y, pred)`、`roc_auc_score(y, proba)`、`ConfusionMatrixDisplay` |
| 查手冊 | <https://scikit-learn.org/stable/>（選對版本）、`函數名稱?`、`.get_params()` |

---
## 延伸閱讀（上課不教）：非監督式學習 – 分群
沒有標籤時，也可以讓電腦自己把住民分成幾群。下面用 K-means，依 ADL、IADL、SPMSQ 把住民分成照護需求相似的 3 群。
期中、期末報告如果要做照護分級、資源配置，可能用得上。

In [ ]:
from sklearn.cluster import KMeans

feat = data[["adl_barthel", "iadl", "spmsq_errors"]]
km = make_pipeline(StandardScaler(), KMeans(n_clusters=3, random_state=1151))
group = km.fit_predict(feat)

print(feat.groupby(group).mean().round(1))          # 每一群的平均
print(pd.Series(group).value_counts().sort_index())  # 每一群的人數